Implementación del detector

In [1]:
from pathlib import Path
import re

ROOT = Path('/Users/angel/computer_vision/lab/practica4')
FRAMES_DIR = ROOT / 'UAV-benchmark-M'
ARTIFACTS = {
    'train': ROOT / 'artifacts' / 'gt_mot_train',
    'test': ROOT / 'artifacts' / 'gt_mot_test',
}
WEIGHTS = {
    'yolo11': ROOT / 'weights' / 'yolo11.pt',
    'yolo26': ROOT / 'weights' / 'yolo26.pt',
    'rtdetr': ROOT / 'weights' / 'rtdetr.pt',
}

COCO_CLASS_NAMES = {2: 'car', 5: 'bus', 7: 'truck'}
DATASET_CLASS_IDS = {'car': 1, 'truck': 2, 'bus': 3}
VEHICLE_CLASSES = set(COCO_CLASS_NAMES.values())
CLASS_ALIASES = {
    'car': 'car', 'cars': 'car',
    'truck': 'truck', 'trucks': 'truck',
    'bus': 'bus', 'buses': 'bus',
}

def frame_number(path):
    match = re.search(r'(\d+)(?=\.[^.]+$)', path.name)
    return int(match.group(1)) if match else None

def sequences_from_artifacts(split):
    gt_dir = ARTIFACTS[split] / 'official_tracking_gt'
    if not gt_dir.is_dir():
        raise FileNotFoundError(f'No existe el directorio de artefactos: {gt_dir}')
    sequences = {}
    for gt_file in sorted(gt_dir.glob('*.txt')):
        sequence = gt_file.stem
        image_dir = FRAMES_DIR / sequence
        images = sorted(
            (p for p in image_dir.glob('*') if p.is_file() and frame_number(p) is not None),
            key=frame_number,
        )
        sequences[sequence] = images
    return sequences

sequences = {split: sequences_from_artifacts(split) for split in ARTIFACTS}
print({split: len(items) for split, items in sequences.items()})


{'train': 30, 'test': 20}


In [2]:
class Detector:
    def __init__(self, weights, confidence=0.25, image_size=1280, device=None, class_mapping=None):
        self.weights = Path(weights)
        self.confidence = confidence
        self.image_size = image_size
        self.device = device
        self.class_mapping = dict(class_mapping or COCO_CLASS_NAMES)
        self.vehicle_ids = set(self.class_mapping)
        self.model = None
        self._load_error = None

    def _load_model(self):
        if self.model is not None or self._load_error is not None:
            return self.model
        if not self.weights.is_file():
            self._load_error = (
                f'No se encontró el peso del detector: {self.weights}. '
                'Configura Detector con la ruta local de un modelo.'
            )
            return None
        try:
            from ultralytics import YOLO
            self.model = YOLO(str(self.weights))
        except ImportError:
            self._load_error = (
                'Falta Ultralytics. Instálalo en el entorno del notebook con '
                '`pip install ultralytics` y vuelve a ejecutar esta celda.'
            )
        except Exception as error:
            self._load_error = f'No se pudo cargar el modelo: {error}'
        return self.model

    def _class_name(self, class_id, names):
        if class_id in self.class_mapping:
            return self.class_mapping[class_id]
        if isinstance(names, dict):
            name = names.get(class_id, names.get(str(class_id), class_id))
        else:
            name = names[class_id] if class_id < len(names) else class_id
        return CLASS_ALIASES.get(str(name).lower(), str(name).lower())

    def predict(self, image_path):
        image_path = Path(image_path)
        frame = frame_number(image_path)
        model = self._load_model()
        if model is None:
            print(self._load_error)
            return []

        results = model.predict(
            source=str(image_path), conf=self.confidence, imgsz=self.image_size,
            device=self.device, verbose=False,
        )
        detections = []
        result = results[0]
        names = getattr(result, 'names', getattr(model, 'names', {}))
        if result.boxes is None:
            return detections
        boxes = result.boxes.xyxy.cpu().tolist()
        confidences = result.boxes.conf.cpu().tolist()
        class_ids = result.boxes.cls.cpu().tolist()
        for bbox, confidence, raw_class_id in zip(boxes, confidences, class_ids):
            class_id = int(raw_class_id)
            if class_id not in self.vehicle_ids:
                continue
            detections.append({
                'frame': frame,
                'bbox': [float(value) for value in bbox],
                'conf': float(confidence),
                'class_id': class_id,
                'class_name': self._class_name(class_id, names),
            })
        return detections


In [3]:
def to_mot_rows(detections):
    rows = []
    for detection in detections:
        x1, y1, x2, y2 = detection['bbox']
        rows.append([
            detection['frame'], -1, x1, y1, x2 - x1, y2 - y1,
            detection['conf'], DATASET_CLASS_IDS[detection['class_name']], 1,
        ])
    return rows


In [4]:
detector = Detector(WEIGHTS['yolo11'])
first_images = sequences['test']
first_image = next((images[0] for images in first_images.values() if images), None)
test_detections = detector.predict(first_image) if first_image else []
test_mot_rows = to_mot_rows(test_detections)
print(f'Imagen de prueba: {first_image}')
print(f'Detecciones: {len(test_detections)} | Filas MOT: {len(test_mot_rows)}')


No se encontró el peso del detector: /Users/angel/computer_vision/lab/practica4/weights/yolo11.pt. Configura Detector con la ruta local de un modelo.
Imagen de prueba: /Users/angel/computer_vision/lab/practica4/UAV-benchmark-M/M0203/img000001.jpg
Detecciones: 0 | Filas MOT: 0
